In [ ]:
import holidays
import datetime as dt
from pyspark.sql import SparkSession
import pyspark.sql.functions as psf
from pyspark.sql.types import BooleanType

spark: SparkSession = (
    SparkSession.builder.config(
        "spark.driver.extraJavaOptions", "-Djava.security.manager=allow"
    )
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow")
    .getOrCreate()
)

In [ ]:
df_visitors = spark.read.parquet("../data/visitors.parquet")

In [ ]:
@psf.udf(returnType=BooleanType())
def is_weekend(timestamp: dt.datetime) -> bool:
    """
    Determine whether a given timestamp falls on a weekend (Saturday or Sunday).
    """

    return timestamp.weekday() in [5, 6]


@psf.udf(returnType=BooleanType())
def is_holiday(timestamp: dt.datetime) -> bool:
    """
    Determine whether a given timestamp falls on a holiday in Belgium.
    """
    return timestamp.date() in holidays.country_holidays("BE")


# OR
# is_holiday_udf = psf.udf(is_holiday, BooleanType())
# is_weekend_udf = psf.udf(is_weekend, BooleanType())

In this exercise, we want to figure out which non-weekend and non-holiday days attracted the most visitors. To do so, we'll write two user-defined functions:


In [ ]:
(
    df_visitors.withColumns(
        {
            "is_weekend": is_weekend("timestamp"),
            "is_holiday": is_holiday("timestamp"),
        }
    )
    .filter(~psf.col("is_weekend") & ~psf.col("is_holiday"))
    .sort(psf.desc("visitors"))
    .show(5, False)
)